# CIFAR-10 Image Classification (No CNN) with Early Stopping
Simple Dense Neural Network trained on the CIFAR-10 dataset.
Training automatically stops once **training accuracy reaches 90%**.

## Step 1: Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

## Step 2: Load the CIFAR-10 Dataset

In [ ]:
# CIFAR-10 dataset (tfds.image_classification.Cifar10) loaded via tf.keras.datasets
cifar10 = tf.keras.datasets.cifar10

(training_images, training_labels), (test_images, test_labels) = cifar10.load_data()

# Labels come as shape (N, 1); flatten them to (N,) for convenience
training_labels = training_labels.flatten()
test_labels = test_labels.flatten()

## Step 3: Preview a Sample Image

In [ ]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

index = 100
print(f'LABEL: {training_labels[index]} ({class_names[training_labels[index]]})')
plt.imshow(training_images[index])
plt.axis('off')
plt.show()

## Step 4: Normalize the Pixel Values

In [ ]:
# Scale pixel values from 0-255 to 0-1
training_images = training_images / 255.0
test_images = test_images / 255.0

## Step 5: Build the Model (Dense Layers Only, No CNN)

In [ ]:
model = Sequential()
model.add(Flatten(input_shape=(32, 32, 3)))
model.add(Dense(512, activation='relu'))
model.add(Dense(256, activation='relu'))
model.add(Dense(10, activation='softmax'))

## Step 6: Compile the Model

In [ ]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

## Step 7: Define Early Stopping Callback (Stop at 90% Accuracy)

In [ ]:
class AccuracyEarlyStopping(tf.keras.callbacks.Callback):
    """Stops training once training accuracy reaches 90%."""

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        accuracy = logs.get('accuracy')

        if accuracy is not None and accuracy >= 0.90:
            print(f"\nAccuracy reached {accuracy * 100:.2f}% (>=90%), stopping training!")
            self.model.stop_training = True


early_stopping = AccuracyEarlyStopping()

## Step 8: Train the Model

In [ ]:
history = model.fit(
    training_images,
    training_labels,
    epochs=200,
    validation_split=0.2,
    callbacks=[early_stopping]
)

## Step 9: Evaluate the Model on Test Data

In [ ]:
test_loss, test_accuracy = model.evaluate(test_images, test_labels)
print(f'Test Accuracy: {test_accuracy * 100:.2f}%')